# Prepare qmaps for sarek+oncoanalyzer results postprocessing

In [1]:
import json
import pandas as pd
import os

Replace `<root>` by the path where you have saved the repository.

## Load sample dictionary and pts list

In [2]:
exclude_pts = ['pt2','pt10','pt16','pt20']
sjd_dict = json.load(open('../../sample_ids/sample_sjd_ids.json','rb'))
sjd_dict = {pt:sjd_dict[pt] for pt in sjd_dict.keys() if pt not in exclude_pts}
print(len(sjd_dict))

target_dict = json.load(open('../../sample_ids/sample_target_ids.json','rb'))
# get patients with bam
target_samples_list = []
for n in range(1,8):
    df1 = pd.read_csv('../../variant_calling/target_cohort/sarek_results/input_fastq_batch'+str(n)+'.csv')
    p = list(df1['patient'].unique())
    target_samples_list = target_samples_list + p
target_dict = {pt:target_dict[pt] for pt in target_dict.keys() if pt in target_samples_list}
print(len(target_dict))

stjude_dict = json.load(open('../../sample_ids/sample_stjude_ids.json','rb'))
bad_quality_samples = ['SJBT032047_D1','SJBT033133_D3']
xenograft = ['SJMRT063832_X1']
no_age_data = ['SJMRT014754_D1']
exclude_samples =  bad_quality_samples + xenograft+ no_age_data
stjude_dict = {pt:stjude_dict[pt] for pt in stjude_dict if stjude_dict[pt]['tumor1'] not in exclude_samples}
print(len(stjude_dict))

pmc_dict = json.load(open('../../sample_ids/sample_pmc_ids.json','rb'))

samples_dict = sjd_dict | target_dict | stjude_dict | pmc_dict
pts = samples_dict.keys()

16
56
16


In [3]:
#all patients
len(pts)

89

In [4]:
tumors_dict = {'tumor1':'1','tumor2':'2','tumor3':'t3','tumor4':'t4','clone1':'c1','clone2':'c2'}

## Create folder tree for output


### Folders for the somatic calling

In [22]:
#get folder names main results

results_folders = ['filter_and_annot',
 'process_cnv',
 'process_sv',
 'process_vep_output',
 'vcf_processing',
 'vep_processing']

In [23]:
#get folder names vcf processing
vcf_folders = ['intersect', 'mutect', 'sage', 'strelka']

In [24]:
#get folder names cnv processing
cnv_folders = ['purple']

In [25]:
#get folder names sv processing
sv_folders = ['gridds']

In [26]:
all_folders = []
path = '../../mafs/'
all_folders.append(path)
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'mafs_sjd'
    elif pt.startswith('PA'):
        cohort = 'mafs_target'
    elif pt.startswith('SJ'):
        cohort = 'mafs_stjude'
    else:
        cohort = 'mafs_pmc'
    cohort_path = path + cohort + '/'
    all_folders.append(cohort_path)
    pt_path = cohort_path + pt +'/'
    all_folders.append(pt_path)
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    for tumor in tumors:
        normal = samples_dict[pt]['normal']
        tumor = samples_dict[pt][tumor]
        if tumor == 'NAN':
            pass
        else:
            tumor_folder = tumor + '_vs_' + normal
            t_pt_path = pt_path + tumor_folder + '/'
            all_folders.append(t_pt_path)    
    
        for results_folder in results_folders:
            results_t_pt_path = t_pt_path + results_folder + '/'
            all_folders.append(results_t_pt_path)
            if results_folder == 'vcf_processing':
                for vcf_folder in vcf_folders:
                    vcf_results_t_pt_path = results_t_pt_path + vcf_folder + '/'
                    all_folders.append(vcf_results_t_pt_path)

            elif results_folder == 'process_cnv':
                cnv_results_t_pt_path = results_t_pt_path + 'purple' + '/'
                all_folders.append(cnv_results_t_pt_path)
            elif results_folder == 'process_sv':
                sv_results_t_pt_path = results_t_pt_path + 'gridds' + '/'
                all_folders.append(sv_results_t_pt_path)            
print(len(all_folders))
all_folders  

1570


['../../mafs/',
 '../../mafs/mafs_sjd/',
 '../../mafs/mafs_sjd/pt1/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/filter_and_annot/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_cnv/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_cnv/purple/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_sv/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_sv/gridds/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_vep_output/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/intersect/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/mutect/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/',
 '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vep_processing/',
 '../../mafs/mafs_sjd/pt1/AX4937_vs_AX4894/',
 '../../mafs/mafs_sjd/pt1/AX4937_vs_AX4894/filter_and_annot/',
 '../..

Run this code once

In [13]:
for folder in all_folders:
    try:
        os.mkdir(folder)
    except FileExistsError:
        pass

### Folders for the germline calling

In [10]:
germline_folders = ['vep_processing',
 'process_vep_output',
 'vcf_processing',
 'filter_and_annot',
 'process_sv']

In [11]:
vcf_folders = ['mutect', 'haplotype_caller']

In [12]:
vep1_folders = ['haplotype_caller']

In [13]:
vep2_folders = ['haplotype_caller', 'sage_germline']

In [14]:
filt_folders = ['haplotype_caller']

In [28]:
all_folders = []
path = '../../mafs/'
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'mafs_sjd'
    elif pt.startswith('PA'):
        cohort = 'mafs_target'
    elif pt.startswith('SJ'):
        cohort = 'mafs_stjude'
    else:
        cohort = 'mafs_pmc'
    cohort_path = path + cohort + '/'
    all_folders.append(cohort_path)
    pt_path = cohort_path + pt +'/'
    normal = samples_dict[pt]['normal']

    n_pt_path = pt_path + normal + '/'
    all_folders.append(n_pt_path)  

    for germline_folder in germline_folders:
        germline_n_pt_path = n_pt_path + germline_folder + '/'
        all_folders.append(germline_n_pt_path)
        
        if germline_folder == 'process_sv':
            vcf_germline_n_pt_path = germline_n_pt_path + 'gripss/'
            all_folders.append(vcf_germline_n_pt_path)
        else:

            vcf_germline_n_pt_path = germline_n_pt_path + 'haplotype_caller/'
            all_folders.append(vcf_germline_n_pt_path)
           
print(len(all_folders))

all_folders

1068


['../../mafs/mafs_sjd/',
 '../../mafs/mafs_sjd/pt1/AX4894/',
 '../../mafs/mafs_sjd/pt1/AX4894/vep_processing/',
 '../../mafs/mafs_sjd/pt1/AX4894/vep_processing/haplotype_caller/',
 '../../mafs/mafs_sjd/pt1/AX4894/process_vep_output/',
 '../../mafs/mafs_sjd/pt1/AX4894/process_vep_output/haplotype_caller/',
 '../../mafs/mafs_sjd/pt1/AX4894/vcf_processing/',
 '../../mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/',
 '../../mafs/mafs_sjd/pt1/AX4894/filter_and_annot/',
 '../../mafs/mafs_sjd/pt1/AX4894/filter_and_annot/haplotype_caller/',
 '../../mafs/mafs_sjd/pt1/AX4894/process_sv/',
 '../../mafs/mafs_sjd/pt1/AX4894/process_sv/gripss/',
 '../../mafs/mafs_sjd/',
 '../../mafs/mafs_sjd/pt3/AX4895/',
 '../../mafs/mafs_sjd/pt3/AX4895/vep_processing/',
 '../../mafs/mafs_sjd/pt3/AX4895/vep_processing/haplotype_caller/',
 '../../mafs/mafs_sjd/pt3/AX4895/process_vep_output/',
 '../../mafs/mafs_sjd/pt3/AX4895/process_vep_output/haplotype_caller/',
 '../../mafs/mafs_sjd/pt3/AX4895/vcf_proces

Run this code once

In [12]:
for folder in all_folders:
    try:
        os.mkdir(folder)
    except FileExistsError:
        pass

## QMAP for process Mutect vcf files

In [30]:
#commands for process vcfs from HMF

input_sage = '../../variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/sage/somatic/AX4912.sage.somatic.filtered.vcf.gz'
input_mutect = '../../variant_calling/sjd_cohort/sarek_results/variant_calling/mutect2/AX4912_vs_AX4894/AX4912_vs_AX4894.mutect2.filtered.vcf.gz'
output_dir = '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/'

root_in = '/<root>/variant_calling/sjd_cohort/' # define root paths
root_out = '/<root>/mafs/mafs_sjd/' #define root paths

python_file = '/<root>/code/python_scripts/process_mutect_vcf.py'

commands = []

#Mutect
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    elif pt.startswith('PA'):
        cohort = 'target'
    else:
        cohort = 'pmc'
    root_in_cohort = '/<root>/variant_calling/'+cohort+'_cohort/'
    root_out_cohort = '/<root>/variant_calling/mafs_'+cohort+'/'

    if pt == 'PMC01':
        root_in_cohort = root_in_cohort.replace('_cohort','_case')
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]
        t = tumors_dict[tumor]
        sample = tumor_id + '_vs_' + normal_id
        t = tumors_dict[tumor]
        in_file = os.path.join(root_in_cohort,'sarek_results','variant_calling','mutect2',sample,sample+'.mutect2.filtered.vcf.gz')
        out_dir = os.path.join(root_out_cohort,pt,sample,'vcf_processing','mutect'+'/')
        normal_id = pt +'_'+ normal_id
        tumor_id = pt +'_'+ tumor_id       
        command = 'python ' + python_file + ' -i ' + in_file + ' -o ' + out_dir + ' -t_id ' + tumor_id + ' -n_id ' + normal_id
        commands.append(command)

commands

['python /<root>/code/python_scripts/process_mutect_vcf.py -i /<root>/variant_calling/sjd_cohort/sarek_results/variant_calling/mutect2/AX4912_vs_AX4894/AX4912_vs_AX4894.mutect2.filtered.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/mutect/ -t_id pt1_AX4912 -n_id pt1_AX4894',
 'python /<root>/code/python_scripts/process_mutect_vcf.py -i /<root>/variant_calling/sjd_cohort/sarek_results/variant_calling/mutect2/AX4937_vs_AX4894/AX4937_vs_AX4894.mutect2.filtered.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/mutect/ -t_id pt1_AX4937 -n_id pt1_AX4894',
 'python /<root>/code/python_scripts/process_mutect_vcf.py -i /<root>/variant_calling/sjd_cohort/sarek_results/variant_calling/mutect2/AX4914_vs_AX4895/AX4914_vs_AX4895.mutect2.filtered.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt3/AX4914_vs_AX4895/vcf_processing/mutect/ -t_id pt3_AX4914 -n_id pt3_AX4895',
 'python /<root>/code/python_scripts/process_mutect_vcf.py -i /<root>/var

In [31]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 20G','[jobs]']
qmap_file = qmap_pre_params + commands

In [32]:
#Save qmap file

with open('01_mutect_process.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## QMAP for process SAGE vcf files

In [33]:
#commands for process vcfs from HMF

input_sage = '../../variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/sage/somatic/AX4912.sage.somatic.filtered.vcf.gz'
input_mutect = '../../variant_calling/sjd_cohort/sarek_results/variant_calling/mutect2/AX4912_vs_AX4894/AX4912_vs_AX4894.mutect2.filtered.vcf.gz'
output_dir = '../../mafs/mafs_sjd/mafs_sarek_oncoa/pt1/AX4912_vs_AX4894/vcf_processing/sage/'

root_in = '/<root>/variant_calling/' # define root paths
root_out = '/<root>/mafs/' #define root paths

python_file = '/<root>/code/python_scripts/process_sage_vcf.py'

commands = []

#SAGE
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    elif pt.startswith('PA'):
        cohort = 'target'
    else:
        cohort = 'pmc'
    root_in_cohort = '/<root>/variant_calling/'+cohort+'_cohort/'
    root_out_cohort = '/<root>/variant_calling/mafs_'+cohort+'/'

    if pt == 'PMC01':
        root_in_cohort = root_in_cohort.replace('_cohort','_case')
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]
        t = tumors_dict[tumor]
        sample = tumor_id + '_vs_' + normal_id
        pt_folder = pt+'_'+t
        in_file = os.path.join(root_in_cohort,'oncoanalyser_results','output',pt_folder,'sage','somatic',tumor_id+'.sage.somatic.vcf.gz')
        out_dir = os.path.join(root_out_cohort,pt,sample,'vcf_processing','sage'+'/')
        command = 'python ' + python_file + ' -i ' + in_file + ' -o ' + out_dir + ' -t_id ' + tumor_id + ' -n_id ' + normal_id
        commands.append(command)    
commands

['python /<root>/code/python_scripts/process_sage_vcf.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/sage/somatic/AX4912.sage.somatic.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/ -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_sage_vcf.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_2/sage/somatic/AX4937.sage.somatic.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/sage/ -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_sage_vcf.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt3_1/sage/somatic/AX4914.sage.somatic.vcf.gz -o /<root>/variant_calling/mafs_sjd/pt3/AX4914_vs_AX4895/vcf_processing/sage/ -t_id AX4914 -n_id AX4895',
 'python /<root>/code/python_scripts/process_sage_vcf.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt3_2/sage/somatic/AX4915.sage.somatic.vcf.gz -o /<r

In [34]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 20G','[jobs]']
qmap_file = qmap_pre_params + commands

In [35]:
#Save qmap file

with open('02_sage_process.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## QMAP file for process strelka vcf

In [36]:
#commands for process vcfs from HMF

input_strelka = '../../variant_calling/sjd_cohort/sarek_results/sjd_variant_calling/strelka/AX4912_vs_AX4894/AX4912_vs_AX4894.strelka.somatic_snvs.vcf.gz'
output_dir = '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/'

root_in = '/<root>/variant_calling/' # define root paths
root_out = '/<root>/mafs/' #define root paths

python_file = '/<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py'

commands = []

#Strelka
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'
    if cohort == 'pmc':
        path_in = root_in + cohort + '_case/'
    else:
        path_in = root_in + cohort + '_cohort/'
    path_out = root_out + 'mafs_' + cohort + '/'
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]

        t = tumors_dict[tumor]
        sample_id = tumor_id + '_vs_' + normal_id
        in_file = os.path.join(path_in,'variant_calling','strelka',sample_id,sample_id+'.strelka.somatic_snvs.vcf.gz')
        out_dir = os.path.join(path_out,pt,sample_id,'vcf_processing','strelka'+'/')
        command = 'python ' + python_file + ' -i ' + in_file + ' -o ' + out_dir + ' -t_id ' + tumor_id + ' -n_id ' + normal_id
        commands.append(command)
commands

['python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4912_vs_AX4894/AX4912_vs_AX4894.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/ -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4937_vs_AX4894/AX4937_vs_AX4894.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/strelka/ -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4914_vs_AX4895/AX4914_vs_AX4895.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/vcf_processing/strelka/ -t_id AX4914 -n_id AX4895',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/

In [37]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 20G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 20G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4912_vs_AX4894/AX4912_vs_AX4894.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/ -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4937_vs_AX4894/AX4937_vs_AX4894.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/strelka/ -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_strelka_v2.9.10_vcf.py -i /<root>/variant_calling/sjd_cohort/variant_calling/strelka/AX4914_vs_AX4895/AX4914_vs_AX4895.strelka.somatic_snvs.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/vcf_processing/strelka/ -t_id AX49

In [38]:
#Save qmap file

with open('03_strelka_process.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## QMAP for intersect by chromosome (HMF)

In [42]:
#commands for process vcfs from HMF

input_sage = '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/AX4912_vs_AX4894_process.maf.gz'
input_mutect = '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/mutect/AX4912_vs_AX4894_process.maf.gz'
input_strelka = '../../mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/AX4912_vs_AX4894_process.maf.gz'

root = '/<root>/mafs/'

python_file = '<root>/code/python_scripts/intersect_callers.py'

commands = []

for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'
    path = root + 'mafs_' + cohort + '/'
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]

        sample_id = tumor_id + '_vs_' + normal_id

        in_file_sage = os.path.join(path,pt,sample_id,'vcf_processing','sage',sample_id+'_process.maf.gz')
        in_file_mutect = os.path.join(path,pt,sample_id,'vcf_processing','mutect',sample_id+'_process.maf.gz')
        in_file_strelka = os.path.join(path,pt,sample_id,'vcf_processing','strelka',sample_id+'_process.maf.gz')
        out_dir = os.path.join(root,pt,sample_id,'vcf_processing','intersect'+'/')
        command = 'python ' + python_file + ' -sa ' + in_file_sage + ' -mu ' + in_file_mutect + ' -st ' + in_file_strelka + ' -o ' + out_dir + ' -sn ' + sample_id + ' -c '
        commands.append(command)
commands

['python <root>/code/python_scripts/intersect_callers.py -sa /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/AX4912_vs_AX4894_process.maf.gz -mu /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/mutect/AX4912_vs_AX4894_process.maf.gz -st /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/AX4912_vs_AX4894_process.maf.gz -o /<root>/mafs/pt1/AX4912_vs_AX4894/vcf_processing/intersect/ -sn AX4912_vs_AX4894 -c ',
 'python <root>/code/python_scripts/intersect_callers.py -sa /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/sage/AX4937_vs_AX4894_process.maf.gz -mu /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/mutect/AX4937_vs_AX4894_process.maf.gz -st /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/strelka/AX4937_vs_AX4894_process.maf.gz -o /<root>/mafs/pt1/AX4937_vs_AX4894/vcf_processing/intersect/ -sn AX4937_vs_AX4894 -c ',
 'python <root>/code/python_scripts/intersect_callers.py -sa /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX489

In [43]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 20G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 20G',
 '[jobs]',
 'python <root>/code/python_scripts/intersect_callers.py -sa /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/sage/AX4912_vs_AX4894_process.maf.gz -mu /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/mutect/AX4912_vs_AX4894_process.maf.gz -st /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/strelka/AX4912_vs_AX4894_process.maf.gz -o /<root>/mafs/pt1/AX4912_vs_AX4894/vcf_processing/intersect/ -sn AX4912_vs_AX4894 -c ',
 'python <root>/code/python_scripts/intersect_callers.py -sa /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/sage/AX4937_vs_AX4894_process.maf.gz -mu /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/mutect/AX4937_vs_AX4894_process.maf.gz -st /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/strelka/AX4937_vs_AX4894_process.maf.gz -o /<root>/mafs/pt1/AX4937_vs_AX4894/vcf_pro

In [44]:
#Save qmap file

with open('04_intersect.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

### QMAP for VEP v.111 HMF somatic intersect

In [93]:
gnomad_url = '/path/to/gnomad_v4.0/'
mafs_url = '<root>/mafs/'
sing_vep_url = '/path/to/ensembl-vep_111.0.sif'
#run vep command

chroms = list(range(1,23))
chroms = ['chr'+str(chrom) for chrom in chroms]
sex_chroms = ['chrX','chrY']
chroms = chroms + sex_chroms

vep_commands = []
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in['normal','sex','normal2']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]
        sample_id = tumor_id + '_vs_' + normal_id
        for chrom in chroms:
            pt_url = os.path.join(mafs_url,'mafs_'+cohort,pt,sample_id,'vcf_processing','intersect',sample_id + '_' + chrom + '.maf.gz')
            chr_file = 'gnomad.genomes.v4.0.sites.'+chrom+'.vcf.bgz'
            gnomad_chr_file = os.path.join(gnomad_url,chr_file)
            out_file = pt_url.replace('vcf_processing/intersect/', 'vep_processing/')
            out_file = out_file.replace('.maf.gz', '_vep.tsv')
            vep_command = sing_vep_url+' vep -i '+ pt_url + ' --format vcf -o '+out_file+' -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom ' + gnomad_chr_file + ',gnomADg,vcf,exact,0,AF,NFE'
            vep_commands.append(vep_command)        
vep_commands

['/path/to/ensembl-vep_111.0.sif vep -i <root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/intersect/AX4912_vs_AX4894_chr1.maf.gz --format vcf -o <root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vep_processing/AX4912_vs_AX4894_chr1_vep.tsv -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom /path/to/gnomad_v4.0/gnomad.genomes.v4.0.sites.chr1.vcf.bgz,gnomADg,vcf,exact,0,AF,NFE',
 '/path/to/ensembl-vep_111.0.sif vep -i <root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/intersect/AX4912_vs_AX4894_chr2.maf.gz --format vcf -o <root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vep_processing/AX4912_vs_AX4894_chr2_vep.tsv -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom /path/to/gnomad_v4.0/gnomad.genomes.v4.0.sites.chr2.vcf.bgz,gnomADg,vcf,exact,0,AF,NFE',
 '/path/to/ensembl-vep_111.0.sif vep -i <root>/mafs/mafs_sjd/pt1/A

In [94]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate vep111','[params]','cores = 1','memory = 8G','[jobs]']
qmap_file = qmap_pre_params + vep_commands
qmap_file
with open('05_vep_gnomad_v4.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## QMAP for process_vep111.py

In [47]:
#run process_vep command
mafs_url = '/<root>/mafs/'
process_vep_commands = []
for pt in pts:
    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex']]

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]

        sample_id = tumor_id + '_vs_' + normal_id
        maf_url = os.path.join(mafs_url,'mafs_'+cohort,pt,sample_id,'vcf_processing','intersect/')
        vep_url = os.path.join(mafs_url,'mafs_'+cohort,pt,sample_id,'vep_processing/')
        command = 'python /<root>/code/python_scripts/process_vep111.py --path_input_vep '+vep_url+' --path_input_maf '+maf_url+' --file_name '+sample_id +' --cores 8'
        process_vep_commands.append(command)
    
    
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 8','memory = 15G','[jobs]']
qmap_file = qmap_pre_params + process_vep_commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 8',
 'memory = 15G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_vep111.py --path_input_vep /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vep_processing/ --path_input_maf /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/vcf_processing/intersect/ --file_name AX4912_vs_AX4894 --cores 8',
 'python /<root>/code/python_scripts/process_vep111.py --path_input_vep /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vep_processing/ --path_input_maf /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/vcf_processing/intersect/ --file_name AX4937_vs_AX4894 --cores 8',
 'python /<root>/code/python_scripts/process_vep111.py --path_input_vep /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/vep_processing/ --path_input_maf /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/vcf_processing/intersect/ --file_name AX4914_vs_AX4895 --cores 8',
 'python /<root>/code/python_scripts/process_vep111.py --path_input_vep /<root

In [48]:
with open('06_process_vep.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## QMAP for filter_and_annot.py

Before running this, it needs to be calculated the ccf thresholds.  
In another notebook:  
/ccf_thresholds/TMB_and_CCF_analysis.ipynb

In [60]:
#run process_vep command
tumors_dict = {'tumor1':'1','tumor2':'2','tumor3':'3','tumor4':'4','clone1':'1','clone2':'2'}

ccf_json_sjd_path = '../../ccf_thresholds/ccf_thresholds_sjd.json'
ccf_json_clones_path = '../../ccf_thresholds/ccf_thresholds_sjd_clones.json'
ccf_json_target_path = '../../ccf_thresholds/ccf_thresholds_target.json'
ccf_json_stjude_path = '../../ccf_thresholds/ccf_thresholds_stjude.json'
ccf_json_pmc_path = '../../ccf_thresholds/ccf_thresholds_pmc_case.json'
ccf_sjd_dict = json.load(open(ccf_json_sjd_path,'rb'))
ccf_clones_dict = json.load(open(ccf_json_clones_path,'rb'))
ccf_target_dict = json.load(open(ccf_json_target_path,'rb'))
ccf_stjude_dict = json.load(open(ccf_json_stjude_path,'rb'))
ccf_pmc_dict = json.load(open(ccf_json_pmc_path,'rb'))
ccf_dict = ccf_sjd_dict | ccf_clones_dict | ccf_target_dict | ccf_stjude_dict | ccf_pmc_dict
purity_sjd_json_path = '../../purities/purities_sjd.json'
purity_target_json_path = '../../purities/purities_target.json'
purity_stjude_json_path = '../../purities/purities_stjude.json'
purity_pmc_json_path = '../../purities/purities_pmc_case.json'
purity_sjd_dict = json.load(open(purity_sjd_json_path, 'rb'))
purity_target_dict = json.load(open(purity_target_json_path, 'rb'))
purity_stjude_dict = json.load(open(purity_stjude_json_path, 'rb'))
purity_pmc_dict = json.load(open(purity_pmc_json_path, 'rb'))
purity_dict = purity_sjd_dict | purity_target_dict | purity_stjude_dict | purity_pmc_dict
process_vep_commands = []
for pt in pts:
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['sex','normal','normal2']]

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        folder = cohort + '_case'
    else:
        folder = cohort + '_cohort' 

    oncoanalyser_results_url = '/<root>/variant_calling/'+folder+'/oncoanalyser_results/output/'
    mafs_url = '/<root>/mafs/mafs_'+cohort+'/'

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]

        sample_id = tumor_id + '_vs_' + normal_id
        t = tumors_dict[tumor]
        if pt.startswith('pt'):
            if 'clone' in tumor:
                ccf = ccf_dict[pt +'_'+tumor]
                purity = 1
            else:
                ccf = ccf_dict[pt +'_t'+t]
                purity = purity_dict[pt +'_t'+t]                

        elif pt.startswith('PA') or pt.startswith('SJ'):
            ccf = ccf_dict[pt]
            purity = purity_dict[pt]
        else:
            ccf = ccf_dict[tumor_id]

        vep_url = os.path.join(mafs_url,pt,sample_id,'process_vep_output/')
        cnv_url = oncoanalyser_results_url + pt +'_'+t+'/purple/'+tumor_id+'.purple.cnv.somatic.tsv'
        output_url = os.path.join(mafs_url,pt,sample_id,'filter_and_annot/')
        script =  '/<root>/code/python_scripts/filter_and_annot_muts.py'
        command = 'python '+script+' -i '+vep_url+' -o '+output_url+' -t_id '+tumor_id + ' -n_id ' +normal_id + ' -ccf '+str(ccf) + ' -c intersect' + ' -cnv ' +cnv_url + ' -pur '+str(purity)
        process_vep_commands.append(command)
    
    
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 8','memory = 15G','[jobs]']
qmap_file = qmap_pre_params + process_vep_commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 8',
 'memory = 15G',
 '[jobs]',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_vep_output/ -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/filter_and_annot/ -t_id AX4912 -n_id AX4894 -ccf 0.45354463823518265 -c intersect -cnv /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/purple/AX4912.purple.cnv.somatic.tsv -pur 0.92',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/process_vep_output/ -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/filter_and_annot/ -t_id AX4937 -n_id AX4894 -ccf 0.6042979399714826 -c intersect -cnv /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_2/purple/AX4937.purple.cnv.somatic.tsv -pur 0.8',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt3/AX491

In [61]:
with open('07_filter_and_annot.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

# Process vcf from GRIDDS/Purple (SV, HMF pipeline)

In [88]:
python_file = '/<root>/code/python_scripts/process_gridds.py'
genomic_positions = '/<root>/data/ensembl111_genomic_regions_mane.tsv.gz'
mane_transcripts = '/<root>/data/MANE.GRCh38.v1.2.summary.txt.gz'

commands = []
for pt in pts:
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['normal','sex','normal2']]

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        vc_folder = 'pmc_case'
    else:
        vc_folder = cohort+'_cohort'

    maf_folder = 'mafs_'+cohort

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]
        if tumor_id == 'NAN':
            pass
        else:
            sample_id = tumor_id + '_vs_' + normal_id
            t = tumors_dict[tumor]

            if cohort == 'pmc':
                oa_folder = pt+'0'+t
            else:
                oa_folder = pt+'_'+t

            input_vcf = '/<root>/variant_calling/oncoanalyser_results/output/'+oa_folder+'/purple/'+tumor_id+'.purple.sv.vcf.gz'
            output_dir = '/<root>/mafs/'+maf_folder+'/'+pt+'/'+sample_id+'/process_sv/gridds/'
            command = 'python '+python_file+' -i '+input_vcf+' -o '+output_dir+' -gp '+genomic_positions+' -mt '+mane_transcripts+' -t_id '+tumor_id+' -n_id '+normal_id
            commands.append(command)
commands

['python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt1_1/purple/AX4912.purple.sv.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_sv/gridds/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt1_2/purple/AX4937.purple.sv.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/process_sv/gridds/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt3_1/purple/AX4914.purple.sv.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/process_sv/gridds/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.

In [89]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 8G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 8G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt1_1/purple/AX4912.purple.sv.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_sv/gridds/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt1_2/purple/AX4937.purple.sv.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/process_sv/gridds/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/oncoanalyser_results/output/pt3_1/purple/AX4914.purple.sv.vcf.gz -o /<root>/mafs/

In [90]:
#Save qmap file

with open('08_process_gridds.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## Process cnv vcf from purple (HMF pipeline)

In [85]:
python_file = '/<root>/code/python_scripts/process_cnv_purple.py'

commands = []
for pt in pts:
    samples = list(samples_dict[pt].keys())
    tumors = [s for s in samples if s not in ['sex','normal','normal2']]

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        vc_folder = 'pmc_case'
    else:
        vc_folder = cohort+'_cohort'

    maf_folder = 'mafs_'+cohort

    for tumor in tumors:
        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt][tumor]
        sample_id = tumor_id + '_vs_' + normal_id
        t = tumors_dict[tumor]
        input_tsv = '/<root>/variant_calling/'+vc_folder+'oncoanalyser_results/output/'+pt+'_'+t+'/purple/'+tumor_id+'.purple.cnv.gene.tsv'
        output_dir = '/<root>/mafs/'+maf_folder+'/'+pt+'/'+sample_id+'/process_cnv/purple/'
        command = 'python '+python_file+' -i '+input_tsv+' -o '+output_dir+' -t_id '+tumor_id+' -n_id '+normal_id
        commands.append(command)
commands

['python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt1_1/purple/AX4912.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_cnv/purple/ -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt1_2/purple/AX4937.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/process_cnv/purple/ -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt3_1/purple/AX4914.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/process_cnv/purple/ -t_id AX4914 -n_id AX4895',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt3_2/purple/AX4915.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt3/AX4915_vs_AX4895/process_cnv/purp

In [86]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 8G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 8G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt1_1/purple/AX4912.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt1/AX4912_vs_AX4894/process_cnv/purple/ -t_id AX4912 -n_id AX4894',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt1_2/purple/AX4937.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt1/AX4937_vs_AX4894/process_cnv/purple/ -t_id AX4937 -n_id AX4894',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_calling/sjd_cohortoncoanalyser_results/output/pt3_1/purple/AX4914.purple.cnv.gene.tsv -o /<root>/mafs/mafs_sjd/pt3/AX4914_vs_AX4895/process_cnv/purple/ -t_id AX4914 -n_id AX4895',
 'python /<root>/code/python_scripts/process_cnv_purple.py -i /<root>/variant_c

In [87]:
#Save qmap file

with open('09_process_cnv_purple.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

# Germline

## Haplotype caller vcf files

In [70]:
#commands for process vcfs from HMF

input_haplotypecaller = '../variant_calling/haplotypecaller/AX4894/AX4894.haplotypecaller.filtered.vcf.gz'
output_dir = '/workspace/projects/rhabdoid_tumors/mafs_sarek_oncoa/pt1/AX4894/vcf_processing/haplotype_caller/'



python_file = '/<root>/code/python_scripts/process_haplotype_caller.py'

commands = []

#Haplotype caller
for pt in pts:

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        vc_folder = 'pmc_case'
    else:
        vc_folder = cohort+'_cohort'

    maf_folder = 'mafs_'+cohort
    
    normal = samples_dict[pt]['normal']

    root_in = '/<root>/variant_calling/'+vc_folder+'/sarek_results/'+cohort+'variant_calling/haplotypecaller/'
    root_out = '/<root>/mafs/'+maf_folder

    in_file = os.path.join(root_in,normal,normal+'.haplotypecaller.filtered.vcf.gz')
    out_dir = os.path.join(root_out,pt,normal,'vcf_processing','haplotype_caller'+'/')
    command = 'python ' + python_file + ' -i ' + in_file + ' -o ' + out_dir + ' -n_id ' +pt+'_'+ normal + ' -c 8'
    commands.append(command)
    
commands

['python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4894/AX4894.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/ -n_id pt1_AX4894 -c 8',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4895/AX4895.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4895/vcf_processing/haplotype_caller/ -n_id pt3_AX4895 -c 8',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4896/AX4896.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt4/AX4896/vcf_processing/haplotype_caller/ -n_id pt4_AX4896 -c 8',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant

In [71]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 8','memory = 80G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 8',
 'memory = 80G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4894/AX4894.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/ -n_id pt1_AX4894 -c 8',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4895/AX4895.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4895/vcf_processing/haplotype_caller/ -n_id pt3_AX4895 -c 8',
 'python /<root>/code/python_scripts/process_haplotype_caller.py -i /<root>/variant_calling/sjd_cohort/sarek_results/sjdvariant_calling/haplotypecaller/AX4896/AX4896.haplotypecaller.filtered.vcf.gz -o /<root>/mafs/mafs_sjd/pt4/AX4896/vcf_processing/haplotype_caller/ -n_

In [72]:
#Save qmap file

with open('10_haplocall_germline_process.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## Vep v.111 on HaplotypeCaller

In [74]:
gnomad_url = '/path/to/gnomad/'
mafs_url = '/<root>/mafs/'
sing_vep_url = '/<root>/code/singularity/ensembl-vep_111.0.sif'
#run vep command

chroms = list(range(1,23))
chroms = ['chr'+str(chrom) for chrom in chroms]
sex_chroms = ['chrX','chrY']
chroms = chroms + sex_chroms

vep_commands = []
for pt in pts:
    normal_id = samples_dict[pt]['normal']

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        vc_folder = 'pmc_case'
    else:
        vc_folder = cohort+'_cohort'

    maf_folder = 'mafs_'+cohort
    
    for chrom in chroms:
        pt_url = os.path.join(mafs_url,maf_folder,pt,normal_id,'vcf_processing','haplotype_caller',normal_id + '_' + chrom + '.maf.gz')
        chr_file = 'gnomad.genomes.v4.0.sites.'+chrom+'.vcf.bgz'
        gnomad_chr_file = os.path.join(gnomad_url,chr_file)
        out_file = pt_url.replace('vcf_processing/intersect/', 'vep_processing/')
        out_file = out_file.replace('.maf.gz', '_vep.tsv')
        vep_command = sing_vep_url+' vep -i '+ pt_url + ' --format vcf -o '+out_file+' -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom ' + gnomad_chr_file + ',gnomADg,vcf,exact,0,AF,NFE'
        vep_commands.append(vep_command)        
vep_commands

['/<root>/code/singularity/ensembl-vep_111.0.sif vep -i /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr1.maf.gz --format vcf -o /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr1_vep.tsv -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom /path/to/gnomad/gnomad.genomes.v4.0.sites.chr1.vcf.bgz,gnomADg,vcf,exact,0,AF,NFE',
 '/<root>/code/singularity/ensembl-vep_111.0.sif vep -i /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr2.maf.gz --format vcf -o /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr2_vep.tsv -tab --assembly GRCh38 --no_stats --cache --symbol --protein --canonical --offline --mane --af_1kg --dir /workspace/datasets/vep  --custom /path/to/gnomad/gnomad.genomes.v4.0.sites.chr2.vcf.bgz,gnomADg,vcf,exact,0,AF,NFE',
 '/<root>/code/singularity/ensembl-vep_111.0.sif vep -i /<root>/mafs/maf

In [75]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 20G','[jobs]']
qmap_file = qmap_pre_params + vep_commands
qmap_file
with open('11_vep_gnomad_v4_hc.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## Process vep output from HaplotypeCaller (HMF)

In [77]:
chroms = list(range(1,23))
chroms = ['chr'+str(chrom) for chrom in chroms]
sex_chroms = ['chrX','chrY']
chroms = chroms + sex_chroms

#run process_vep command
mafs_url = '/<root>/mafs/'
python_file = '/<root>/code/python_scripts/process_vep111_haplocall.py'
process_vep_commands = []
for pt in pts:

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    maf_folder = 'mafs_'+cohort
    
    for chrom in chroms:
        normal = samples_dict[pt]['normal']
        vep_file_name = normal + '_' + chrom + '_vep.tsv'
        maf_file_name = normal + '_' + chrom + '.maf.gz'
        maf_url = os.path.join(mafs_url,maf_folder,pt,normal,'vcf_processing','haplotype_caller',maf_file_name)
        vep_url = os.path.join(mafs_url,maf_folder,pt,normal,'vep_processing','haplotype_caller',vep_file_name)
        command = 'python '+python_file+' --path_input_vep '+vep_url+' --path_input_maf '+maf_url+' -c 24'
        process_vep_commands.append(command)

qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 24','memory = 100G','[jobs]']
qmap_file = qmap_pre_params + process_vep_commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 24',
 'memory = 100G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_vep111_haplocall.py --path_input_vep /<root>/mafs/mafs_sjd/pt1/AX4894/vep_processing/haplotype_caller/AX4894_chr1_vep.tsv --path_input_maf /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr1.maf.gz -c 24',
 'python /<root>/code/python_scripts/process_vep111_haplocall.py --path_input_vep /<root>/mafs/mafs_sjd/pt1/AX4894/vep_processing/haplotype_caller/AX4894_chr2_vep.tsv --path_input_maf /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr2.maf.gz -c 24',
 'python /<root>/code/python_scripts/process_vep111_haplocall.py --path_input_vep /<root>/mafs/mafs_sjd/pt1/AX4894/vep_processing/haplotype_caller/AX4894_chr3_vep.tsv --path_input_maf /<root>/mafs/mafs_sjd/pt1/AX4894/vcf_processing/haplotype_caller/AX4894_chr3.maf.gz -c 24',
 'python /<root>/code/

In [78]:
with open('12_process_vep_haplocall.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## Haplotype caller: Filter and prepare

In [ ]:
python_file = '/<root>/code/python_scripts/filter_and_annot_muts.py'

commands = []
for pt in pts:

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'#canonical_transcripts = '/workspace/users/msanchezg/notebooks/protein_degradation/table_files/ensembl_canonical_transcripts.tsv'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    maf_folder = 'mafs_'+cohort
    
    normal = samples_dict[pt]['normal']
    input_dir =  '/<root>/mafs/'+maf_folder+'/'+pt+'/'+normal+'/process_vep_output/haplotype_caller/'
    output_dir = '/<root>/mafs/'+maf_folder+'/'+pt+'/'+normal+'/filter_and_annot/haplotype_caller/'
    command = 'python '+python_file+' -i '+input_dir+' -o '+output_dir+' -n_id '+normal+' -c hc -gn 0.01'
    commands.append(command)
commands

['python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt1/AX4894/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt1/AX4894/filter_and_annot/haplotype_caller/ -n_id AX4894 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt3/AX4895/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt3/AX4895/filter_and_annot/haplotype_caller/ -n_id AX4895 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt4/AX4896/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt4/AX4896/filter_and_annot/haplotype_caller/ -n_id AX4896 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt5/AX4897/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt5/AX4897/filter_and_annot/haplotype_caller/ -n_id AX4897 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_mu

In [80]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 8G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 8G',
 '[jobs]',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt1/AX4894/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt1/AX4894/filter_and_annot/haplotype_caller/ -n_id AX4894 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt3/AX4895/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt3/AX4895/filter_and_annot/haplotype_caller/ -n_id AX4895 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt4/AX4896/process_vep_output/haplotype_caller/ -o /<root>/mafs/mafs_sjd/pt4/AX4896/filter_and_annot/haplotype_caller/ -n_id AX4896 -c hc -gn 0.01',
 'python /<root>/code/python_scripts/filter_and_annot_muts.py -i /<root>/mafs/mafs_sjd/pt5/AX4897/process_vep_output/haplotype_caller/ -o /<root>

In [81]:
with open('13_filter_and_annot_germline.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)

## Process vcf from GRIPSS, germline SV

In [82]:
python_file = '/<root>/code/python_scripts/process_gridds.py'
genomic_positions = '/<root>/data/ensembl111_genomic_regions_mane.tsv.gz'
mane_transcripts = '/<root>/data/MANE.GRCh38.v1.2.summary.txt.gz'

commands = []
for pt in pts:
    samples = list(samples_dict[pt].keys())

    if pt.startswith('pt'):
        cohort = 'sjd'
    elif pt.startswith('PA'):
        cohort = 'target'
    elif pt.startswith('SJ'):
        cohort = 'stjude'
    else:
        cohort = 'pmc'

    if cohort == 'pmc':
        vc_folder = 'pmc_case'
    else:
        vc_folder = cohort+'_cohort'

    maf_folder = 'mafs_'+cohort

    if 'normal' in samples_dict[pt].keys():

        normal_id = samples_dict[pt]['normal']
        tumor_id = samples_dict[pt]['tumor1']

        input_vcf = '/<root>/variant_calling/'+vc_folder+'/oncoanalyser_results/output/'+pt+'_1/purple/'+tumor_id+'.purple.sv.germline.vcf.gz'
        output_dir = '/<root>/mafs/'+maf_folder+'/'+pt+'/'+normal_id+'/process_sv/gripss/'
        command = 'python '+python_file+' -i '+input_vcf+' -o '+output_dir+' -gp '+genomic_positions+' -mt '+mane_transcripts+' -n_id '+normal_id +' --is_germline'
        commands.append(command)
commands

['python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/purple/AX4912.purple.sv.germline.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4894/process_sv/gripss/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -n_id AX4894 --is_germline',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt3_1/purple/AX4914.purple.sv.germline.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4895/process_sv/gripss/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -n_id AX4895 --is_germline',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt4_1/purple/AX4916.purple.sv.germline.vcf.gz -o /<root>/mafs/mafs_sjd/pt4/AX4896/process_sv/gripss/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root

In [83]:
qmap_pre_params = ['[pre]','. "/home/$USER/miniconda3/etc/profile.d/conda.sh"','conda activate rhabdoids','[params]','cores = 1','memory = 8G','[jobs]']
qmap_file = qmap_pre_params + commands
qmap_file

['[pre]',
 '. "/home/$USER/miniconda3/etc/profile.d/conda.sh"',
 'conda activate rhabdoids',
 '[params]',
 'cores = 1',
 'memory = 8G',
 '[jobs]',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt1_1/purple/AX4912.purple.sv.germline.vcf.gz -o /<root>/mafs/mafs_sjd/pt1/AX4894/process_sv/gripss/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -n_id AX4894 --is_germline',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt3_1/purple/AX4914.purple.sv.germline.vcf.gz -o /<root>/mafs/mafs_sjd/pt3/AX4895/process_sv/gripss/ -gp /<root>/data/ensembl111_genomic_regions_mane.tsv.gz -mt /<root>/data/MANE.GRCh38.v1.2.summary.txt.gz -n_id AX4895 --is_germline',
 'python /<root>/code/python_scripts/process_gridds.py -i /<root>/variant_calling/sjd_cohort/oncoanalyser_results/output/pt4_1/purple/AX4916.

In [84]:
#Save qmap file
with open('14_process_gripss.qmap', 'w') as f:
    for item in qmap_file:
        f.write('%s\n' % item)